In [31]:
import pandas as pd

raw = pd.read_csv("club_signups.csv", dtype=str)
df = raw.copy()
print(df.shape)
print(df.dtypes)
df.head()


(39, 9)
signup_id       str
student_id      str
full_name       str
email           str
faculty         str
city            str
club            str
signed_up_at    str
fee_paid        str
dtype: object


,signup_id,student_id,full_name,email,faculty,city,club,signed_up_at,fee_paid
0,S015,61-8942,Hana Zaki,hana.zaki@student.guc.edu.eg,Pharmacy,Giza,Soccer,2026-09-15 13:39,N
1,S010,61-3846,AHMED ZAKI,ahmed.zaki@student.guc.edu.eg,Management,alexandria,Chess,15/09/2026 03:12,false
2,S008,61-4844,Mohamed Adel,mohamed.adel@student.guc.edu.eg,EMS,Cairo,Music,2026-09-14 23:32,Y
3,S035,61-4844,MOHAMED ADEL,mohamed.adel@student.guc.edu.eg,Engineering and Materials Science,Cairo,Debating,2026-09-17 11:15,yes
4,S004,61-7344,Laila Mostafa,laila.mostafa@student.guc.edu.eg,IET,Alexandria,Chess Club,2026-09-13 22:36,yes


In [32]:
categorical_cols = ["faculty", "city", "club", "fee_paid"]

for col in categorical_cols:
    print(df[col].value_counts(dropna=False))

faculty
Pharmacy                                  7
EMS                                       7
IET                                       5
Management                                4
Information Engineering and Technology    4
MET                                       3
iet                                       3
Engineering and Materials Science         1
Pharma                                    1
met                                       1
Media Engineering and Technology          1
pharmacy                                  1
ems                                       1
Name: count, dtype: int64
city
Cairo         13
Alexandria    11
Giza           4
Alex           4
cairo          2
giza           2
alexandria     1
CAIRO          1
El Giza        1
Name: count, dtype: int64
club
Robotics       8
Music          5
Chess          4
football       4
Debating       3
Chess Club     3
Football       3
Soccer         2
music          2
chess          2
debate club    1
Music          1
D

In [33]:
faculty = df["faculty"].str.strip().str.lower()
faculty.value_counts()

faculty
pharmacy                                  8
ems                                       8
iet                                       8
management                                4
met                                       4
information engineering and technology    4
engineering and materials science         1
pharma                                    1
media engineering and technology          1
Name: count, dtype: int64

In [34]:
FACULTY_MAP = {
    "met": "MET",
    "media engineering and technology": "MET",
    "iet": "IET",
    "information engineering and technology": "IET",
    "ems": "EMS",
    "engineering and materials science": "EMS",
    "pharmacy": "Pharmacy",
    "pharma": "Pharmacy",
    "management": "Management"
}

df["faculty"] = faculty.map(FACULTY_MAP)
df["faculty"].value_counts()

faculty
IET           12
Pharmacy       9
EMS            9
MET            5
Management     4
Name: count, dtype: int64

In [35]:
club = df["club"].str.strip().str.lower()
club.value_counts()

club
music          8
robotics       8
football       7
chess          6
debating       3
chess club     3
soccer         2
debate club    1
debate         1
Name: count, dtype: int64

In [36]:
CLUB_MAP = {
    "music": "Music",
    "robotics": "Robotics",
    "football": "Football",
    "soccer": "Football",
    "chess": "Chess",
    "chess club": "Chess",
    "debating": "Debate",
    "debate club": "Debate",
    "debate": "Debate"
}

df["club"] = club.map(CLUB_MAP)
df["club"].value_counts()

club
Football    9
Chess       9
Music       8
Robotics    8
Debate      5
Name: count, dtype: int64

In [37]:
city = df["city"].str.strip().str.lower()
city.value_counts()

city
cairo         16
alexandria    12
giza           6
alex           4
el giza        1
Name: count, dtype: int64

In [38]:
CITY_MAP = {
    "cairo": "Cairo",
    "alexandria": "Alexandria",
    "alex": "Alexandria",
    "giza": "Giza",
    "el giza": "Giza"
}

df["city"] = city.map(CITY_MAP)
df["city"].value_counts()

city
Alexandria    16
Cairo         16
Giza           7
Name: count, dtype: int64

In [39]:
fee_paid = df["fee_paid"].str.strip().str.lower()
fee_paid.value_counts()

fee_paid
yes      11
1         7
false     6
y         5
n         4
true      4
0         1
no        1
Name: count, dtype: int64

In [40]:
FEE_MAP = {
    "yes": True,
    "y": True,
    "true": True,
    "1": True,
    "no": False,
    "n": False,
    "false": False,
    "0": False
}

df["fee_paid"] = fee_paid.map(FEE_MAP)
df["fee_paid"].value_counts()

fee_paid
True     27
False    12
Name: count, dtype: int64

In [41]:
assert set(df["faculty"]).issubset({"MET", "IET", "EMS", "Pharmacy", "Management"})
assert set(df["club"]).issubset({"Robotics", "Debate", "Music", "Football", "Chess"})
assert set(df["city"]).issubset({"Cairo", "Giza", "Alexandria"})
assert set(df["fee_paid"]).issubset({True, False})

In [42]:
df["full_name"] = (
    df["full_name"]
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
    .str.title()
)

df["email"] = (
    df["email"]
    .str.strip()
    .str.lower()
)

In [43]:
df["fee_paid"] = df["fee_paid"].astype(bool)
print(df["fee_paid"].dtype)

bool


In [44]:
df["signed_up_at"].head()

0    2026-09-15 13:39
1    15/09/2026 03:12
2    2026-09-14 23:32
3    2026-09-17 11:15
4    2026-09-13 22:36
Name: signed_up_at, dtype: str

In [45]:
s = df["signed_up_at"].str.strip()
is_iso = s.str.match(r"^\d{4}-\d{2}-\d{2} \d{2}:\d{2}$")
is_slash = s.str.match(r"^\d{2}/\d{2}/\d{4} \d{2}:\d{2}$")
print("ISO:", is_iso.sum(), "| slash:", is_slash.sum())

ISO: 25 | slash: 14


In [46]:
dmy = s[is_slash]

print("first part > 12:", (dmy.str[:2].astype(int) > 12).sum())
print("second part > 12:", (dmy.str[3:5].astype(int) > 12).sum())

first part > 12: 14
second part > 12: 0


In [47]:
df["signed_up_at"] = pd.to_datetime(
    df["signed_up_at"],
    format="mixed",
    dayfirst=True
)

df["signed_up_at"].head()

0   2026-09-15 13:39:00
1   2026-09-15 03:12:00
2   2026-09-14 23:32:00
3   2026-09-17 11:15:00
4   2026-09-13 22:36:00
Name: signed_up_at, dtype: datetime64[us]

In [48]:
print("Rows before removing exact duplicates:", len(df))
n_before = len(df)
exact = df.duplicated().sum()
print(f"Exact duplicate rows: {exact}")
df[df.duplicated(keep=False)]

Rows before removing exact duplicates: 39
Exact duplicate rows: 3


,signup_id,student_id,full_name,email,faculty,city,club,signed_up_at,fee_paid
0,S015,61-8942,Hana Zaki,hana.zaki@student.guc.edu.eg,Pharmacy,Giza,Football,2026-09-15 13:39:00,False
15,S022,61-8224,Sara Zaki,sara.zaki@student.guc.edu.eg,EMS,Giza,Music,2026-09-16 01:39:00,True
20,S023,58-1125,Ziad Saleh,ziad.saleh@student.guc.edu.eg,IET,Cairo,Chess,2026-09-16 09:29:00,False
24,S022,61-8224,Sara Zaki,sara.zaki@student.guc.edu.eg,EMS,Giza,Music,2026-09-16 01:39:00,True
33,S023,58-1125,Ziad Saleh,ziad.saleh@student.guc.edu.eg,IET,Cairo,Chess,2026-09-16 09:29:00,False
34,S015,61-8942,Hana Zaki,hana.zaki@student.guc.edu.eg,Pharmacy,Giza,Football,2026-09-15 13:39:00,False


In [49]:
df = df.drop_duplicates()
print("Rows after removing exact duplicates:", len(df))
assert df.duplicated().sum() == 0
assert len(df) == n_before - exact

Rows after removing exact duplicates: 36


In [51]:
n_before = len(df)

same_student_club = df.duplicated(
    subset=["student_id", "club"],
    keep=False
)

print("Rows involved in student + club duplicates:", same_student_club.sum())

display(df[same_student_club].sort_values(["student_id", "club", "signed_up_at"]))

Rows involved in student + club duplicates: 8


,signup_id,student_id,full_name,email,faculty,city,club,signed_up_at,fee_paid
8,S020,55-9245,Ali Ibrahim,ali.ibrahim@student.guc.edu.eg,MET,Cairo,Music,2026-09-15 23:07:00,False
9,S033,55-9245,Ali Ibrahim,ali.ibrahim@student.guc.edu.eg,MET,Cairo,Music,2026-09-16 09:07:00,True
22,S021,61-4844,Mohamed Adel,mohamed.adel@student.guc.edu.eg,EMS,Cairo,Debate,2026-09-16 01:15:00,False
3,S035,61-4844,Mohamed Adel,mohamed.adel@student.guc.edu.eg,EMS,Cairo,Debate,2026-09-17 11:15:00,True
35,S017,61-7344,Laila Mostafa,laila.mostafa@student.guc.edu.eg,IET,Alexandria,Debate,2026-09-15 18:01:00,False
25,S034,61-7344,Laila Mostafa,laila.mostafa@student.guc.edu.eg,IET,Alexandria,Debate,2026-09-16 02:01:00,True
21,S032,61-7344,Laila Mostafa,laila.mostafa@student.guc.edu.eg,IET,Alexandria,Football,2026-09-17 23:36:00,False
38,S036,61-7344,Laila Mostafa,laila.mostafa@student.guc.edu.eg,IET,Alexandria,Football,2026-09-18 23:36:00,True


In [52]:
df = df.sort_values("signed_up_at")
df = df.drop_duplicates(
    subset=["student_id", "club"],
    keep="last"
)

In [53]:
n_after = len(df)
removed = n_before - n_after
print("Rows removed:", removed)
print("Rows after removing student + club duplicates:", n_after)
assert not df.duplicated(["student_id", "club"]).any()

Rows removed: 4
Rows after removing student + club duplicates: 32


In [54]:
df.to_csv("club_signups_clean.csv", index=False)
print("Saved:", "club_signups_clean.csv")

Saved: club_signups_clean.csv


In [ ]:
check = pd.read_csv("club_signups_clean.csv")
print(check.shape)
print(check.dtypes)

(32, 9)
signup_id        str
student_id       str
full_name        str
email            str
faculty          str
city             str
club             str
signed_up_at     str
fee_paid        bool
dtype: object
